# 01. 카드 데이터 전처리

**읽는 것**: `data/raw/card/` 데이터1.txt (시간대), 데이터2.txt (고객 거주지)

**만드는 것**: `data/processed/panel_time.csv`, `panel_visitor.csv`, `docs/industry_mapping.csv`

런타임 → 세션 다시 시작 후 모두 실행으로 처음부터 끝까지 돌리면 됩니다.

## 1. 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
RAW_CARD = BASE / "data/raw/card"
EXT = BASE / "data/external"
OUT = BASE / "data/processed"
DOCS = BASE / "docs"
OUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

## 2. 컬럼명 규칙

| 원본 | 변경 | 설명 |
|---|---|---|
| TA_YMD | date | 날짜 |
| TIME_GB | time_gb | 시간대 (데이터1만) |
| MCT_SGG_CD | region | 강남 / 춘천 |
| MCT_RY_CD | industry | 원본 업종 |
| CLN_SGG_CD | cust_sido | 고객 거주 시도 (데이터2만) |
| SEX_CCD | sex | 성별 |
| AGE_CCD | age | 연령대 |
| TS_AT | sales | 결제금액 |
| USE_CNT | cnt | 결제건수 |

In [ ]:
CARD_RENAME = {"TA_YMD": "date", "TIME_GB": "time_gb", "MCT_SGG_CD": "region",
               "MCT_RY_CD": "industry", "CLN_SGG_CD": "cust_sido",
               "SEX_CCD": "sex", "AGE_CCD": "age", "TS_AT": "sales", "USE_CNT": "cnt"}
REGION = {"서울 강남구": "강남", "강원 춘천시": "춘천"}
HOME = {"강남": "서울", "춘천": "강원"}   # 지역민 기준 시도

## 3. 업종 그룹 매핑

- **제외**: 날씨와 무관한 정기·의무 결제, 사무·전문 서비스, 결제 대행
- **제외_내구재**: 고액·비정기라 하루 매출이 크게 튀는 업종
- 나머지 13개 그룹이 분석 대상

In [ ]:
IND_GROUPS = {
    "제외": ["세금공과금", "ZZ_나머지", "보험", "통신요금(이동,시내전화)",
            "통신요금(PC통신,무선호출)", "상품권/복권", "방문판매/다단계판매",
            "고속버스/철도/여객선", "예식장/결혼서비스", "연구/번역서비스",
            "법률/사무서비스", "회계/변리서비스", "결제대행(PG)", "부동산중개",
            "학교등록금", "유치원", "장례식장/묘지/장의사", "보건소",
            "여행사/항공사", "택시"],
    "제외_내구재": ["컴퓨터/소프트웨어", "가전", "가구", "수입자동차",
                 "인테리어/건축자재/주방기구", "오토바이", "중고차판매"],
    "음식점": ["한식", "중식", "일식", "양식", "기타요식", "패스트푸드"],
    "카페·제과": ["커피전문점", "제과점"],
    "식료품소매": ["할인점/슈퍼마켓/양판점", "편의점", "기타식품", "정육점",
               "농수산물", "주류판매"],
    "패션·잡화": ["의복/의류", "패션잡화", "생활잡화/수입상품점", "화장품",
              "시계/귀금속", "스포츠/레저용품", "문화용품", "완구/아동용자전거",
              "화원", "악기/음반", "수제용품점", "중고품판매점", "사무기기/문구용품"],
    "대형유통": ["백화점", "쇼핑몰", "면세점", "기타유통", "전용매장", "체인점"],
    "의료": ["일반병원", "종합병원", "치과병원", "한의원", "약국", "기타의료"],
    "실내여가": ["영화/공연", "노래방", "게임방/오락실", "싸우나/목욕탕", "유흥업소"],
    "스포츠": ["스포츠시설", "헬스장"],
    "야외레저": ["종합레저타운/놀이동산", "실내/실외골프장"],
    "숙박": ["모텔,여관,기타숙박", "호텔/콘도"],
    "생활서비스": ["미용실", "세탁소", "미용서비스", "안마/마사지", "동물병원", "애완동물"],
    "교육": ["학원/학습지", "서점", "독서실"],
    "자동차": ["주유소", "LPG가스", "자동차서비스", "자동차용품", "주차장"],
}
IND_MAP = {name: g for g, names in IND_GROUPS.items() for name in names}

pd.DataFrame(list(IND_MAP.items()), columns=["industry", "ind_grp"]) \
  .to_csv(DOCS / "industry_mapping.csv", index=False, encoding="utf-8-sig")
print("매핑된 업종 수:", len(IND_MAP))

## 4. 읽기 + 정제

- 법인 제외 (데이터1에만 있음, 금액 약 16%)
- 분석 제외 업종 제거
- 연령 표기 통일: `20 대` → `20대`

In [ ]:
def clean_card(fname):
    df = pd.read_csv(RAW_CARD / fname, sep="\t", encoding="cp949", dtype=str)
    df = df.rename(columns=CARD_RENAME)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df["sales"] = pd.to_numeric(df["sales"])
    df["cnt"] = pd.to_numeric(df["cnt"])
    df["region"] = df["region"].map(REGION)
    df["age"] = df["age"].str.replace(" ", "")
    df["ind_grp"] = df["industry"].map(IND_MAP).fillna("미분류")

    unmapped = sorted(df.loc[df["ind_grp"] == "미분류", "industry"].unique())
    assert not unmapped, f"매핑 안 된 업종: {unmapped}"

    df = df[df["sex"] != "법인"]
    df = df[~df["ind_grp"].str.startswith("제외")]
    return df.reset_index(drop=True)

c1 = clean_card("신한카드_빅콘테스트2026_데이터1.txt")   # 시간대
c2 = clean_card("신한카드_빅콘테스트2026_데이터2.txt")   # 고객 거주지
c2["is_visitor"] = (c2["cust_sido"] != c2["region"].map(HOME)).astype(int)

print("데이터1:", c1.shape, "| 데이터2:", c2.shape)
print(c1.columns.tolist())
print(c2.columns.tolist())

## 5. 분석 단위로 집계 + 저장

| 파일 | 단위 | 비고 |
|---|---|---|
| panel_time.csv | 날짜 × 지역 × 업종그룹 × 시간대 | 00_05 제외 (춘천 일부 결측) |
| panel_visitor.csv | 날짜 × 지역 × 업종그룹 × 외지인여부 | |

In [ ]:
panel_time = (c1[c1["time_gb"] != "00_05"]
              .groupby(["date", "region", "ind_grp", "time_gb"])[["sales", "cnt"]]
              .sum().reset_index())
panel_visitor = (c2.groupby(["date", "region", "ind_grp", "is_visitor"])[["sales", "cnt"]]
                 .sum().reset_index())

panel_time.to_csv(OUT / "panel_time.csv", index=False, encoding="utf-8-sig")
panel_visitor.to_csv(OUT / "panel_visitor.csv", index=False, encoding="utf-8-sig")

print("panel_time:", panel_time.shape, panel_time.columns.tolist())
print("panel_visitor:", panel_visitor.shape, panel_visitor.columns.tolist())

## 6. 점검

In [ ]:
# 업종그룹 × 지역 × 시간대/외지인 조합마다 184일이 다 있는지
print(panel_time.groupby(["region", "ind_grp", "time_gb"])["date"].nunique().unstack([0, 2]).min())
print(panel_visitor.groupby(["region", "ind_grp", "is_visitor"])["date"].nunique().unstack([0, 2]).min())

# 지역별 외지인 매출 비중 (강남 약 46%, 춘천 약 18% 예상)
v = panel_visitor.groupby(["region", "is_visitor"])["sales"].sum().unstack()
print((v[1] / v.sum(axis=1)).round(3))